# Week 2 — How Text Becomes Numbers

**LLMs & You · Hampden-Sydney College · Fall 2026**

On Tuesday we said a model turns text into vectors. This notebook is where you
find out what that actually looks like, by running it.

---

## This is not a worksheet

There are no steps to complete and nothing to hand in on Wednesday night. The
code below works. Running it top to bottom takes about ten minutes and teaches
you very little.

**What you are doing is looking for something to show on Thursday.** Every
section ends with a **Your turn** box: a question the code can answer but I have
not answered for you. Pick the ones you find interesting, change the inputs, and
find a result that surprised you — including, and especially, one where the
method fell over.

Thursday is you showing the room what you found. Ten minutes each, informally.
A finding that broke is worth more than one that worked: the labs are graded on
the write-up, not the result.

!!! note
    You can do all of this without writing code, if you would rather. Paste a
    section into a chatbot and ask it to explain what the output means, or ask
    it to modify the code for a question you have. That is a legitimate way to
    work through this and it is not cheating — it is the tooling half of the
    course. Bring what you found either way.

---

## What you need

Nothing installed. This runs in Google Colab in your browser: **Runtime →
Run all**, or ⇧⏎ through the cells one at a time.

If you would rather run it on your own machine, see
[the setup guide](https://Nalaquq.github.io/llms-and-you/guides/setup/). The
code is identical.

---
## 0. Setup

One cell, once per session. Colab forgets everything when the tab is closed, so
if you come back tomorrow, run this again.

In [ ]:
# Colab has numpy, pandas, matplotlib and scikit-learn already.
# These three are the ones it does not.
%pip install --quiet "gensim>=4.3.3" "transformers>=4.40" nltk

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("numpy", np.__version__)
print("pandas", pd.__version__)

!!! warning "If the gensim import fails later with a numpy error"
    Colab sometimes has a numpy already loaded that gensim will not accept.
    **Runtime → Restart session**, then run this cell again and carry on. You do
    not need to re-run anything above it. This is the single most common thing
    that goes wrong in this notebook and it is not your fault.

---
## 1. A corpus to work on

Everything below needs a **corpus** — a collection of documents. Ours is
deliberately tiny and deliberately about two subjects that share almost no
vocabulary, so you can see the methods working with your own eyes rather than
trusting a number.

Real corpora are millions of documents. Keep that gap in mind: several things
below fail *only* because this corpus is small, and one of the useful findings
you could bring Thursday is working out which ones.

In [ ]:
corpus = [
    # -- coffee --------------------------------------------------------
    "The barista ground the coffee beans fresh for every single cup.",
    "Dark roast coffee beans taste bitter, but the caffeine hits faster.",
    "She drinks her coffee black: no sugar, no milk, every morning.",
    "The cafe roasts its own coffee beans in a small drum roaster.",
    "A good espresso needs fresh coffee beans, hot water, and real pressure.",
    "He ordered a coffee with steamed milk and complained it was scalded.",
    # -- dogs ----------------------------------------------------------
    "The dog barked at the mail carrier again this morning.",
    "A puppy needs training, patience, and a great deal of sleep.",
    "Her dog learned to sit and stay after a week of training.",
    "The vet said the dog needs more exercise and less food.",
    "A tired dog is a good dog, which is why we walk the dog twice.",
    "He adopted a rescue puppy from the shelter and started training it.",
    # -- the ambiguous one ---------------------------------------------
    "The cafe lets you bring your dog inside while you drink your coffee.",
]

labels = ["coffee"] * 6 + ["dog"] * 6 + ["both"]

for i, (doc, lab) in enumerate(zip(corpus, labels)):
    print(f"[{i:2}] ({lab:6}) {doc}")

!!! tip "Your turn — and this one is worth doing early"
    **Replace this corpus with your own** and every result below changes.

    Thirteen short documents about something you actually know: song lyrics,
    Discord messages, your own essays, headlines about one news story, product
    reviews. Two clear topics plus one document that straddles them works best.

    Almost every interesting finding people bring to Thursday comes from a
    corpus they chose. The coffee-and-dogs one is here so the notebook runs, not
    because it is interesting.

---
## 2. Tokenization — before any of this, text has to be cut up

A model never sees letters. It sees **tokens**: pieces of text from a fixed
vocabulary, worked out in advance from a large corpus. Where the cuts fall is
not obvious and it is not per-word.

We are using a real tokenizer — the one BERT ships with — from Hugging Face. It
downloads a small vocabulary file the first time and needs no account and no key.

In [ ]:
from transformers import AutoTokenizer

# BERT's tokenizer. Downloads a small vocabulary file once. No account, no key.
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

for text in ["strawberry", "raspberry", "blackberry", "embeddings",
             "hallucination", "Vaswani", "Hampden-Sydney", "tokenization"]:
    print(f"{text:16} -> {tok.tokenize(text)}")

`##` means "this piece continues the one before it". So `embeddings` is not one
thing to the model — it is `em`, `##bed`, `##ding`, `##s`, and not one of those
four is the word. The name of the author of the paper this course is named after
comes apart into three meaningless fragments.

**Now look at the berries.** `strawberry` is a single token. `blackberry` is a
single token. `raspberry` is three. Nothing about English makes raspberries
different — the vocabulary was built by counting what was frequent in the
training corpus, and strawberries were written about more often. The boundaries
are an artefact of a corpus, not a fact about language.

### Why "how many r's in strawberry" is a hard question

This is the famous one. It is not a reasoning failure. Look at what the model is
actually handed:

In [ ]:
word = "strawberry"
pieces = tok.tokenize(word)
ids = tok.convert_tokens_to_ids(pieces)

print(f"you see:         {word}   ({len(word)} letters)")
print(f"the model sees:  {pieces}")
print(f"which is really: {ids}")
print()
print(f"r's in the string, to you:  {word.count('r')}")
print(f"integers the model receives: {len(ids)}")

The entire word is **one integer: 16876.**

Not three letters and a bag of others — one number, whose relationship to the
letters `s-t-r-a-w-b-e-r-r-y` is a lookup table the model never sees. Asking how
many r's are in it is like asking how many r's are in the number 16876. The
letters were gone before the model started; counting them is not introspection,
it is asking about something the input no longer contains.

A model that answers this correctly has usually learned the *answer* from text
about the question, which is a different thing from being able to look.

### Tokens are not words, and the ratio is not fixed

In [ ]:
samples = {
    "plain English":   "The dog barked at the mail carrier this morning.",
    "technical":       "Tokenization maps substrings to integer identifiers.",
    "numbers":         "The invoice totalled 1,048,576 dollars and 37 cents.",
    "a URL":           "Read it at https://en.wikipedia.org/wiki/Word2vec today.",
    "not English":     "El perro ladró al cartero esta mañana.",
    "made up":         "The flibbertigibbet unhelpfully defenestrated itself.",
}

rows = []
for name, text in samples.items():
    pieces = tok.tokenize(text)
    rows.append({
        "sample": name,
        "characters": len(text),
        "words": len(text.split()),
        "tokens": len(pieces),
        "tokens/word": round(len(pieces) / len(text.split()), 2),
    })

pd.DataFrame(rows).set_index("sample").sort_values("tokens/word")

Plain English costs about 1.1 tokens per word. A URL costs 4.4. The Spanish
sentence costs 1.9 to say the same thing the English one said for 1.1.

If you were being billed per token — and with a commercial model you are — the
same question costs different amounts depending on the language you ask it in.
That is not a pricing decision anybody made deliberately. It falls out of whose
text the vocabulary was built from.

!!! tip "Your turn"
    1. Find text with a **tokens-per-word ratio above 3.0**, and text below 1.2.
       What kind of writing is each? What does that tell you about the corpus the
       vocabulary was built from?
    2. Try a language that is not English — ideally one that does not use the
       Latin alphabet. Compare the ratio to the English sentence saying the same
       thing. **Who pays more to ask the same question?**
    3. `AutoTokenizer.from_pretrained("gpt2")` is a completely different
       tokenizer. Run the same words through both. **A token count from one model
       is simply wrong for another** — this is the most useful practical fact in
       this section, and it is how cost estimates go wrong in production.
    4. Find another arbitrary split like `strawberry` vs `raspberry`: two closely
       related words the tokenizer treats completely differently. Common first
       names work well, and the pattern in *which* names stay whole is worth a
       comment on Thursday.
    5. Find a task that fails for tokenization reasons and is **not** letter
       counting. Rhyming, acrostics, reversing a string, arithmetic on long
       numbers — pick one and work out precisely why token boundaries are the
       problem. This is the most interesting question on the page and the one
       fewest people bring.

---
## 3. Bag of words — the simplest possible number

Take the vocabulary of the whole corpus. For each document, count how many times
each word appears. That row of counts *is* the document, as far as this method is
concerned.

It is called a **bag** of words because a bag has no order. Everything about
sequence is thrown away on the first line.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer()
counts = count_vec.fit_transform(corpus)

vocab = count_vec.get_feature_names_out()

print(f"{len(corpus)} documents")
print(f"{len(vocab)} words in the vocabulary")
print(f"matrix shape: {counts.shape}   (documents x vocabulary)")
print()
print("first 25 of the vocabulary:", list(vocab[:25]))

In [ ]:
# The document-term matrix, as a table you can actually read.
bow = pd.DataFrame(counts.toarray(), columns=vocab)
bow.index = [f"[{i}] {lab}" for i, lab in enumerate(labels)]

# Only the columns that appear more than once, or this is 100+ columns of zeros.
common = bow.columns[bow.sum(axis=0) > 1]
bow[common]

### Most of this matrix is zero

That is not a quirk of our small corpus — it gets *worse* with more data. Every
new document adds words to the vocabulary, so every existing document gains
another zero.

In [ ]:
total = counts.shape[0] * counts.shape[1]
filled = counts.nnz

print(f"cells in the matrix: {total}")
print(f"cells that are not zero: {filled}")
print(f"sparsity: {100 * (1 - filled / total):.1f}% zeros")
print()
print("One document as the model has it:")
print(bow.iloc[0].values)

This is a **sparse vector**: mostly zeros, one dimension per vocabulary word.
Hold onto that — in section 7 we meet dense vectors, which are the opposite, and
the contrast is the whole point of the week.

### What the bag threw away

In [ ]:
pair = ["The dog bit the man.", "The man bit the dog."]

probe = CountVectorizer()
vectors = probe.fit_transform(pair).toarray()

print(pd.DataFrame(vectors, columns=probe.get_feature_names_out(), index=pair))
print()
print("identical vectors:", np.array_equal(vectors[0], vectors[1]))

Two sentences that mean opposite things. One vector. Every method in sections 3
to 6 of this notebook cannot tell them apart, and no amount of extra data fixes
it — the information was discarded by the representation, not lost in the noise.

That limitation is why the rest of the course exists.

!!! tip "Your turn"
    1. Construct your own pair of sentences with **identical bag-of-words vectors
       and opposite meanings**. Harder than it looks — bring the best one.
    2. How large does the vocabulary get if you add ten more documents to the
       corpus? Add them and measure. Does sparsity go up or down?
    3. `CountVectorizer(binary=True)` records *presence* rather than *count*.
       Which of the results further down this notebook change if you use it? Form
       a prediction first, then check.

---
## 4. Preprocessing — every knob here is a decision you are making

`CountVectorizer` quietly did several things in section 3: lowercased everything,
threw away punctuation, split on whitespace. Those are defaults, not laws. Each
one is a decision with consequences.

In [ ]:
def vocab_size(**kwargs):
    v = CountVectorizer(**kwargs)
    v.fit(corpus)
    return len(v.get_feature_names_out()), v

settings = {
    "default":                dict(),
    "keep case":              dict(lowercase=False),
    "drop English stopwords": dict(stop_words="english"),
    "words in >= 2 docs":     dict(min_df=2),
    "bigrams as well":        dict(ngram_range=(1, 2)),
    "bigrams only":           dict(ngram_range=(2, 2)),
}

for name, kw in settings.items():
    n, _ = vocab_size(**kw)
    print(f"{name:24} vocabulary: {n:4}")

### Stop words: the most common words carry the least information

`the`, `a`, `is`, `and`. They appear everywhere, so they distinguish nothing. The
usual move is to drop them.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

print(f"scikit-learn's English stop list has {len(ENGLISH_STOP_WORDS)} words")
print()
print("a sample:", sorted(ENGLISH_STOP_WORDS)[:20])
print()
# Words in our corpus that the stop list would delete:
in_corpus = set(vocab) & ENGLISH_STOP_WORDS
print(f"of our {len(vocab)}-word vocabulary, {len(in_corpus)} are stop words:")
print(sorted(in_corpus))

!!! warning "Look at that list before you accept it"
    `not` is in scikit-learn's stop list. So are `no`, `never`, `nothing`, and
    `cannot`.

    Drop stop words from *"the coffee was not good"* and you get
    *"coffee good"*. If you were building a review classifier, you just deleted
    the review. Stop-word removal is a default that is wrong for entire
    categories of task, and it is applied by people who never look at the list.

In [ ]:
# See it happen.
review = "the coffee was not good and I would never come back"
kept = [w for w in review.split() if w not in ENGLISH_STOP_WORDS]
print("original:", review)
print("kept:    ", " ".join(kept))

### Stemming and lemmatization: collapsing word forms

`roast`, `roasts`, `roasted`, `roasting` are four columns in our matrix and one
idea. Two ways to collapse them, and they are not the same thing.

In [ ]:
import nltk
nltk.download("wordnet", quiet=True)

from nltk.stem import PorterStemmer, WordNetLemmatizer

stem = PorterStemmer().stem
lemma = WordNetLemmatizer().lemmatize

words = ["roasts", "roasting", "roasted", "beans", "better", "running",
         "studies", "geese", "was", "caresses", "university", "universe"]

pd.DataFrame({
    "word": words,
    "stemmed": [stem(w) for w in words],
    "lemmatized (noun)": [lemma(w) for w in words],
    "lemmatized (verb)": [lemma(w, pos="v") for w in words],
}).set_index("word")

Read that table carefully — the last two rows are the point.

**Stemming** chops suffixes by rule. It is fast, it needs no dictionary, and its
output is frequently not a word (`studi`). It also collapses `university` and
`universe` onto the same stem, which is a genuine error you now have in your
data and will never see again.

**Lemmatization** looks the word up and returns the real dictionary form. It is
slower, it needs to know the part of speech to work properly (look at `was`
across the two columns), and it is right more often.

!!! tip "Your turn"
    1. Find a **stemming collision** of your own: two words with unrelated
       meanings that Porter maps to the same stem. `university`/`universe` is the
       classic. There are many.
    2. Rebuild the bag of words with `stop_words="english"` and again with
       `min_df=2`. Does the document similarity in section 6 get better or worse?
       *Better* by what measure — and can you defend it?
    3. Bigrams: `ngram_range=(1,2)` gets you `"dark roast"` as a single feature.
       That recovers a little word order. How much does the vocabulary grow, and
       is the trade worth it on your corpus?

---
## 5. TF-IDF — counts, weighted by how surprising the word is

Raw counts have an obvious problem: `the` appears in every document, so it gets
the biggest number and tells you nothing. TF-IDF fixes that with two halves.

**Term frequency (TF)** — how often the word appears *in this document*. Common
here is important here.

**Inverse document frequency (IDF)** — how rare the word is *across the corpus*.
A word in every document scores near zero. A word in one document scores high.

Multiply them. A word that is frequent in this document and rare everywhere else
gets the highest weight, which is a decent working definition of "what this
document is about".

Let us compute it by hand once, before letting scikit-learn do it.

In [ ]:
# By hand, on the binary counts, so the formula is visible.
n_docs = len(corpus)
doc_freq = (counts.toarray() > 0).sum(axis=0)      # docs containing each word

# scikit-learn's smoothed idf, so our numbers will match theirs below.
idf_by_hand = np.log((1 + n_docs) / (1 + doc_freq)) + 1

idf_table = pd.DataFrame({
    "word": vocab,
    "in how many docs": doc_freq,
    "idf": idf_by_hand.round(3),
}).sort_values("idf")

print("LOWEST idf — appears everywhere, tells you nothing:")
print(idf_table.head(8).to_string(index=False))
print()
print("HIGHEST idf — appears once, highly distinguishing:")
print(idf_table.tail(8).to_string(index=False))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer()
tfidf = tfidf_vec.fit_transform(corpus)

# Check our arithmetic against theirs.
sklearn_idf = pd.Series(tfidf_vec.idf_, index=tfidf_vec.get_feature_names_out())
ours = pd.Series(idf_by_hand, index=vocab)

print("our idf matches scikit-learn's:", np.allclose(sklearn_idf.values, ours.values))

In [ ]:
# What is each document "about", according to TF-IDF?
terms = tfidf_vec.get_feature_names_out()
dense = tfidf.toarray()

for i, doc in enumerate(corpus):
    top = [j for j in np.argsort(dense[i])[::-1][:3] if dense[i][j] > 0]
    words = ", ".join(terms[j] for j in top)
    print(f"[{i:2}] {doc[:46]:48} -> {words}")

### That is not what the textbook promised

Read the right-hand column. `but`. `no`. `in`. `with`. `was`. `this`. `is`.
`your`. Those are not what these documents are about.

TF-IDF is supposed to suppress words like that, and here it has not. **Work out
why before reading on** — the answer is in the IDF table above.

The reason: IDF punishes a word for appearing in *many documents*, and we only
have thirteen. `but` appears in exactly one of them, so it scores as rare and
distinguishing. In a corpus of fifty thousand documents `but` would appear in
most of them and be crushed to nothing — but in a corpus of thirteen it looks
just as special as `barista`.

**TF-IDF's famous behaviour is a large-corpus behaviour.** At this size you have
to help it, which is what section 4 was for:

In [ ]:
# The same thing, with the function words removed first.
tfidf_clean_vec = TfidfVectorizer(stop_words="english")
tfidf_clean = tfidf_clean_vec.fit_transform(corpus)

terms_c = tfidf_clean_vec.get_feature_names_out()
dense_c = tfidf_clean.toarray()

for i, doc in enumerate(corpus):
    top = [j for j in np.argsort(dense_c[i])[::-1][:3] if dense_c[i][j] > 0]
    words = ", ".join(terms_c[j] for j in top)
    print(f"[{i:2}] {doc[:46]:48} -> {words}")

Now it works. `barista, cup`. `vet, food`. `shelter, rescue`. `walk, tired`.
With no labels and no training, the method has found what each document is about.

Two lessons, and the second is the one people miss:

1. TF-IDF really does extract topic — the 1972 idea is sound.
2. **It only did so after a preprocessing decision you made in section 4.** The
   first output was not a bug in TF-IDF; it was TF-IDF working exactly as
   specified on a corpus too small for the specification to bite. Almost every
   "the method does not work" moment you will have this semester is one of
   these two things.

We can even measure how much that decision bought us:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import itertools

def separation(matrix):
    # Mean similarity within a topic vs across topics. A bigger gap is better.
    sim = cosine_similarity(matrix)
    pairs = list(itertools.combinations(range(12), 2))    # exclude doc 12, it is both
    same = np.mean([sim[i, j] for i, j in pairs if labels[i] == labels[j]])
    cross = np.mean([sim[i, j] for i, j in pairs if labels[i] != labels[j]])
    return same, cross

for name, matrix in [("raw TF-IDF", tfidf), ("stop words removed", tfidf_clean)]:
    same, cross = separation(matrix)
    print(f"{name:22} same topic {same:.3f}   cross topic {cross:.3f}   ratio {same/cross:.1f}x")

Same-topic similarity barely moved. Cross-topic similarity fell by about
three-quarters. Removing stop words did not make related documents look more
alike — it stopped unrelated documents looking alike for no reason.

That is a real experimental result, produced in two cells, and it is exactly the
shape of thing worth bringing on Thursday.

!!! tip "Your turn"
    1. Which word in the corpus has the **highest** IDF, and which the lowest?
       Explain both in one sentence each without using the word "idf".
    2. `TfidfVectorizer(sublinear_tf=True)` uses `1 + log(tf)` instead of the raw
       count. Why would anyone want that? *Hint: look at document 10, which says
       "dog" three times.*
    3. Does `min_df=2` improve the separation ratio more or less than
       `stop_words="english"` did? Measure it with the function above. Which
       would you defend, and on what grounds?
    4. Take document 12 — the one about both dogs and coffee. What does TF-IDF
       think it is about? Is that the right answer?
    5. Run all of this on **your own corpus** and look at the top terms. Where it
       is wrong, is it wrong because of the method or because of your
       preprocessing? That distinction is the interesting part.

---
## 6. Cosine similarity — measuring "close together"

Every document is now a vector. Two documents are similar if their vectors point
in the same direction.

We use the **angle** between them, not the distance. A long document and a short
document about the same subject have very different vector lengths but nearly the
same direction, and it is the direction that carries the meaning.

Cosine similarity is 1.0 for identical direction, 0.0 for perpendicular — which,
for word counts, means sharing no words at all.

In [ ]:
# Use the cleaned matrix from section 5 -- we just measured that it is better.
sim = cosine_similarity(tfidf_clean)

short = [f"[{i}] {lab}" for i, lab in enumerate(labels)]
pd.DataFrame(sim.round(2), index=short, columns=short)

In [ ]:
# The diagonal is every document compared with itself, so it is always 1.00.
# Leaving it in makes everything else look black by comparison -- the real
# similarities here top out around 0.24. Drop it so the colour scale spans
# the numbers we actually care about.
masked = sim.copy()
np.fill_diagonal(masked, np.nan)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(masked, cmap="viridis")

ax.set_xticks(range(len(corpus)), [f"{i}\n{l}" for i, l in enumerate(labels)], fontsize=8)
ax.set_yticks(range(len(corpus)), [f"[{i}] {l}" for i, l in enumerate(labels)], fontsize=8)
ax.set_title("TF-IDF cosine similarity, self-comparisons removed\n"
             "(0-5 coffee, 6-11 dog, 12 both)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

Two blocks are visible along the diagonal — but faintly, and you had to be told
they were there. **The strongest similarity between any two different documents
in this corpus is about 0.24.** Documents on the same topic, written to be
obviously on the same topic, are 76% dissimilar by this measure.

That is not a mistake in the plot. It is what word-overlap similarity actually
looks like on short documents: two sentences about dogs share the word "dog" and
almost nothing else, so the score stays low no matter how plainly related they
are to a human reader.

In [ ]:
# Nearest neighbours: for each document, the most similar other document.
for i in range(len(corpus)):
    others = sim[i].copy()
    others[i] = -1                      # never yourself
    j = int(np.argmax(others))
    flag = "  <-- crosses topics" if labels[i] != labels[j] else ""
    print(f"[{i:2}] {labels[i]:6} closest to [{j:2}] {labels[j]:6}  ({others[j]:.2f}){flag}")

Eleven of the twelve single-topic documents find a neighbour on their own topic.
For a method that does nothing but count words, that is better than it has any
right to be — and the two that cross are worth looking at rather than
dismissing.

!!! tip "Your turn"
    1. Two rows do something interesting: one coffee document's nearest
       neighbour is the ambiguous document 12 rather than another coffee
       document, and document 12 itself has to land somewhere. Look at both.
       **Which single shared word caused each?** Print the terms the two
       documents have in common rather than guessing.
    2. Redo the matrix with the raw `tfidf` instead of `tfidf_clean`. Does the
       11/12 get better or worse? Does the *heatmap* get more or less readable?
       These can move in opposite directions.
    3. Redo it with plain `counts` from section 3. Which of the three rankings is
       best, and how would you argue that to someone who disagreed?
    4. Write two sentences that mean the same thing and **share no words**. What
       is their cosine similarity? Do not guess — run it. The answer is the next
       section.

---
## 7. Where counting hits its ceiling

Everything so far treats words as opaque symbols. `dog` and `puppy` are two
unrelated columns. Nothing in the method knows they are related, and nothing in
the corpus can teach it.

In [ ]:
pairs = [
    ("The dog is asleep on the sofa.",  "The puppy is napping on the couch."),
    ("She bought a car this morning.",  "She purchased an automobile today."),
    ("The film was terrible.",          "The movie was awful."),
]

for a, b in pairs:
    plain = TfidfVectorizer().fit_transform([a, b])
    clean = TfidfVectorizer(stop_words="english").fit_transform([a, b])
    print(f"{a}\n{b}")
    print(f"   with function words: {cosine_similarity(plain)[0][1]:.3f}")
    print(f"   content words only:  {cosine_similarity(clean)[0][1]:.3f}")
    print()

Look at the two columns.

The first sentence pair scores **0.50** — which looks like real similarity until
you notice the only words they share are `the`, `is`, and `on`. That number is
measuring English grammar, not meaning. Strip the function words and the truth
appears: **0.000, all three pairs.**

Three pairs of sentences that mean the same thing, scored as having *nothing
whatsoever* in common. `dog`/`puppy`, `car`/`automobile`, `film`/`movie` — to
every method in this notebook so far, those are six unrelated symbols.

You cannot fix this by counting harder, and you cannot fix it with a bigger
corpus. The representation has nowhere to put the fact that `car` and
`automobile` are related, because a column in that matrix is a *string*, and
those are different strings. The fix has to change the representation itself.

That is section 8, and it is the reason the rest of this course exists.

!!! tip "Your turn"
    That 0.50 is worth sitting with. **How much of the similarity you measured in
    section 6 was this same effect?** You have the tool to check — you ran it in
    section 5. Somebody should bring the answer on Thursday.

---
## 8. word2vec — learning meaning from company

> *"You shall know a word by the company it keeps."* — J.R. Firth, 1957

The **distributional hypothesis**: words appearing in similar contexts have
similar meanings. `dog` and `puppy` both turn up near `bark`, `walk`, `vet`,
`adopt`. Nobody has to say they are related — it falls out of the co-occurrence.

word2vec turns that into a training task. Slide a window along a huge corpus and
learn to predict a word from its neighbours (**CBOW**) or its neighbours from the
word (**skip-gram**). The prediction is not the point and gets thrown away; the
weights learned along the way are the vectors we wanted. That trick — a fake task
whose real product is the representation — is called a **pretext task**, and it
is how most of modern machine learning is trained.

The result is a **dense** vector: 50 to 300 numbers, none of them zero, none of
them meaning anything on its own.

We are loading vectors somebody else trained, on six billion words. Downloading
them takes a minute or so.

In [ ]:
import gensim.downloader

# 50-dimensional GloVe, trained on Wikipedia + Gigaword. ~66 MB.
# (GloVe is not word2vec, but it is the same idea and downloads in a minute
#  rather than an hour. Where they differ does not matter at this level.)
wv = gensim.downloader.load("glove-wiki-gigaword-50")

print(f"vocabulary: {len(wv.index_to_key):,} words")
print(f"dimensions: {wv.vector_size}")
print()
print("the vector for 'dog':")
print(wv["dog"].round(2))

Fifty numbers. **None of them means anything by itself** — there is no "animal"
dimension you could point at. The meaning is in the whole vector's position
relative to every other vector, which is why we always measure with similarity
rather than reading off components.

Compare that to section 3, where every column had a name and 98% were zero.

In [ ]:
for word in ["dog", "coffee", "bank", "king", "python"]:
    neighbours = ", ".join(f"{w} ({s:.2f})" for w, s in wv.most_similar(word, topn=6))
    print(f"{word:8} -> {neighbours}")

In [ ]:
# The pairs that scored zero in section 7:
for a, b in [("dog", "puppy"), ("car", "automobile"), ("film", "movie"),
             ("dog", "coffee"), ("dog", "democracy")]:
    print(f"{a:6} ~ {b:12} {wv.similarity(a, b):.3f}")

`car` and `automobile` were 0.00 under TF-IDF. Here they are close, and `dog`
and `democracy` are not. Nobody wrote a thesaurus — it came out of counting what
words appeared near what.

### The famous one, and where it breaks

`king - man + woman ≈ queen`. It is the demonstration that made word2vec famous
in 2013, and it does work:

In [ ]:
def analogy(a, b, c, topn=3):
    # b is to a as c is to ?    ->    king - man + woman -> queen
    return wv.most_similar(positive=[c, a], negative=[b], topn=topn)

for a, b, c in [("king", "man", "woman"), ("paris", "france", "japan"),
                ("walking", "walk", "swim")]:
    got = ", ".join(f"{w} ({s:.2f})" for w, s in analogy(a, b, c))
    print(f"{a} - {b} + {c}  ->  {got}")

In [ ]:
# Now the ones nobody puts in the blog posts.
for a, b, c in [("doctor", "man", "woman"),
                ("boss", "man", "woman"),
                ("dog", "bark", "meow"),
                ("hot", "cold", "tall")]:
    got = ", ".join(f"{w} ({s:.2f})" for w, s in analogy(a, b, c))
    print(f"{a} - {b} + {c}  ->  {got}")

!!! warning "Read that output"
    The vectors learned the corpus, and the corpus was written by people. What
    comes back for `doctor - man + woman` is not a fact about doctors. It is a
    measurement of what was written near those words in Wikipedia and news text
    through 2014, reproduced without comment.

    This is the cleanest demonstration of training-data bias you will meet all
    semester, and it takes one line to run. We come back to it in Week 14.

### One vector per word, forever

word2vec gives each word exactly one vector, no matter how many meanings it has.

In [ ]:
neighbours = wv.most_similar("bank", topn=12)
print("neighbours of 'bank':")
for w, s in neighbours:
    print(f"   {w:14} {s:.2f}")

River bank and money bank, blended into one point. The vector is a compromise
between senses that has to serve every sentence the word ever appears in, and it
is a good vector for neither.

This is **polysemy**, and it is precisely what contextual embeddings — BERT,
and everything the rest of this course is about — were invented to fix. A modern
model gives `bank` a *different* vector in each sentence. That is the single
biggest idea separating Week 2 from Week 3.

!!! tip "Your turn"
    1. Find an analogy that works impressively. Find one that fails absurdly.
       Bring both — the failure is the more interesting slide.
    2. Find another word like `bank` whose neighbour list mixes two meanings.
       `mouse`, `spring`, `crane`, `bat`, `left`. Which is worst, and why that
       one?
    3. Probe the bias further: try job titles, nationalities, adjectives. What is
       the model reproducing? **Be ready to say what you would do about it if this
       were inside something you shipped.**
    4. `wv.doesnt_match(["coffee", "espresso", "latte", "dog"])` picks the odd one
       out. Find a set where it gets it wrong.
    5. Try `wv.similarity("hot", "cold")`. It is *high*. Why would two opposites
       be close together in this space? What does that tell you about what
       "similar" means here? — this catches almost everyone out.

---
## 9. Training your own — and why yours will be bad

The vectors above took a large corpus and real compute. You can run the same
algorithm on our thirteen sentences in under a second. You should, because the
result is instructive.

In [ ]:
from gensim.models import Word2Vec

tokenized = [doc.lower().replace(",", "").replace(".", "").split() for doc in corpus]

tiny = Word2Vec(
    sentences=tokenized,
    vector_size=50,
    window=5,        # the sliding window: how many neighbours count as "context"
    min_count=1,     # keep every word (with 13 sentences we cannot afford to drop any)
    sg=1,            # 1 = skip-gram, 0 = CBOW
    negative=5,      # negative sampling: how many non-neighbours to contrast against
    epochs=100,
    seed=42,
)

print(f"vocabulary: {len(tiny.wv.index_to_key)} words")
print()
print("neighbours of 'dog', trained on our 13 sentences:")
for w, s in tiny.wv.most_similar("dog", topn=6):
    print(f"   {w:14} {s:.2f}")

That is nonsense, and it is *supposed* to be nonsense.

The algorithm is identical to the one that produced the good vectors. The only
difference is that it saw 13 sentences instead of six billion words. The
distributional hypothesis needs distributions, and thirteen sentences do not have
one — the model has no evidence about `dog` beyond a handful of accidents.

**This is the most important cell in the notebook.** Scale is not a detail of
these methods; below a threshold they simply do not work, and the threshold is
much higher than intuition suggests.

In [ ]:
# CBOW vs skip-gram, same data, one flag apart.
cbow = Word2Vec(tokenized, vector_size=50, window=5, min_count=1, sg=0, epochs=100, seed=42)
skip = Word2Vec(tokenized, vector_size=50, window=5, min_count=1, sg=1, epochs=100, seed=42)

print(f"{'':14} {'CBOW':>28} {'skip-gram':>28}")
for word in ["dog", "coffee"]:
    c = ", ".join(w for w, _ in cbow.wv.most_similar(word, topn=3))
    s = ", ".join(w for w, _ in skip.wv.most_similar(word, topn=3))
    print(f"{word:14} {c:>28} {s:>28}")

!!! tip "Your turn"
    1. Change `window` from 2 to 10 and watch the neighbours move. Small windows
       capture *syntactic* similarity (words that could grammatically swap);
       large windows capture *topical* similarity. Can you see that happening?
    2. How much text does it take before this starts producing something
       sensible? Paste in a chapter of a book from
       [Project Gutenberg](https://www.gutenberg.org/) and find out. **Reporting
       roughly where the threshold is would be an excellent Thursday
       contribution.**
    3. `sg=0` versus `sg=1` — which does better on a tiny corpus, and does the
       documentation's explanation of why match what you observe?

---
## 10. Seeing 50 dimensions on a flat screen

You cannot draw a 50-dimensional space. **PCA** squashes it to two, keeping as
much of the spread as it can.

The picture is a lossy summary and you should distrust it slightly — but it does
show real structure.

In [ ]:
from sklearn.decomposition import PCA

words = ["dog", "puppy", "cat", "kitten", "horse", "vet", "bark", "leash",
         "coffee", "espresso", "latte", "tea", "milk", "beans", "cafe", "barista",
         "king", "queen", "man", "woman", "prince", "princess"]

X = np.array([wv[w] for w in words])
flat = PCA(n_components=2, random_state=0).fit_transform(X)

groups = {"animals": slice(0, 8), "drinks": slice(8, 16), "royalty": slice(16, 22)}
colours = {"animals": "tab:orange", "drinks": "tab:brown", "royalty": "tab:purple"}

fig, ax = plt.subplots(figsize=(10, 7))
for name, sl in groups.items():
    ax.scatter(flat[sl, 0], flat[sl, 1], s=90, c=colours[name], label=name, alpha=0.75)
for i, w in enumerate(words):
    ax.annotate(w, (flat[i, 0], flat[i, 1]), fontsize=10,
                xytext=(5, 4), textcoords="offset points")

ax.set_title("GloVe vectors, 50 dimensions squashed to 2 by PCA")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

!!! tip "Your turn"
    1. Add your own word list and look for structure the picture reveals — or
       claims to reveal that you do not believe.
    2. Two words can look adjacent here and be far apart in the real 50
       dimensions. **Find a pair where the picture lies**, by checking
       `wv.similarity` against what you see. This is a genuinely useful habit:
       every 2-D embedding plot you will ever see in a paper has this problem.
    3. Where do `king`, `queen`, `man`, `woman` sit relative to each other? Is
       the parallelogram from section 8 visible?

---
## What to bring on Thursday

Pick **one or two** things you found. Not everything — ten minutes each,
informally, laptop open at the cell.

A finding is a claim plus the evidence you ran. It fits in three sentences:

> *I expected X. I ran Y. I got Z, which surprised me because…*

Good ones from this notebook, roughly in order of how interesting they usually
turn out to be:

| | |
|:---|:---|
| **A method that broke** | A stemming collision, an analogy that failed absurdly, a case where the PCA picture lies, a nearest-neighbour result that is plainly wrong |
| **Your own corpus** | Text you chose, run through TF-IDF. Did the top terms match what those documents are actually about? |
| **A tokenization failure** | Something that fails for token-boundary reasons and is *not* letter counting |
| **A threshold** | How much text word2vec needs before it stops producing nonsense |
| **Bias you found** | With the line you ran, and what you would do about it if this were in production |
| **A disagreement** | Somewhere the notebook's framing, or mine, seems wrong to you |

!!! success "You do not need working code to have something to show"
    A screenshot, a table you pasted into a doc, a chatbot conversation where you
    worked out what an output meant, or a question the notebook raised and did
    not answer — all fine. **A confusion you can state precisely is a
    contribution.** Half of what makes Thursday useful is someone saying the
    thing several other people were quietly stuck on.

---

### Where this goes next

| Concept | Where it goes |
|:---|:---|
| One vector per word, senses blended | **Week 3** — attention gives each occurrence its own vector |
| The sliding window | **Week 3** — attention replaces it with "look at everything at once" |
| Bag of words loses order | **Week 3** — positional encoding puts it back |
| Cosine similarity | **Week 10** — retrieval is this, over a document store |
| Bias in the vectors | **Week 14** — auditing, and who is accountable |

Every concept in this notebook has an entry on the
[study guide](https://Nalaquq.github.io/llms-and-you/study-guide/) with a
definition, what you should be able to do with it, and where to review it.

<small>Questions before Thursday: sgleason@hsc.edu, or office hours T/Th 12–2.</small>